# Multi-Agent System Notebook

This notebook demonstrates the design and orchestration of a multi-agent system, covering agent design, communication protocols, task delegation, and orchestration between multiple collaborating agents.

## 1. Install and Import Required Libraries

Install and import the libraries needed to build the multi-agent system. This example relies only on the Python standard library (`dataclasses`, `enum`, `queue`, `uuid`, `collections`) plus `matplotlib`/`networkx` for visualization, so it runs without external agent frameworks. Uncomment the pip install line if you want to experiment with a framework such as `autogen` or `langchain`.

In [ ]:
# install required packages
# Install these once from a terminal with:
# .\.venv\Scripts\python -m pip install -r requirements.txt
# Then select the "Invoice Manager (.venv)" kernel in Jupyter/VS Code.

## Create pre-trained agent

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_MODEL", "gpt-5"),
)
qa_agent = create_agent(
    llm,
    tools=[],
    system_prompt=(
        "You are an assistant that answers questions. "
        "Use your pre-trained data to answer to the best of your ability. Say you don't know if you are not sure of the answer."
    ),
)

print("QA agent created successfully.")

question = "Who is Allen Huan Bui?"
response = qa_agent.invoke({"messages": [{"role": "user", "content": question}]})

answer_content = response["messages"][-1].content
if isinstance(answer_content, list):
    answer_text = "".join(part.get("text", "") for part in answer_content if isinstance(part, dict))
else:
    answer_text = answer_content

print(f"Q: {question}")
print(f"A: {answer_text}")

QA agent created successfully.
Q: Who is Allen Huan Bui?
A: I’m not aware of a widely recognized public figure named Allen Huan Bui. It may be a private individual or I might need more context. Could you share any details (field, organization, location, or why you’re asking) so I can try to help identify the right person?


## Create RAG agent (Allen Bui)

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

DATA_SOURCE = "Allen-Bio.txt"

@tool
def read_allen_bio() -> str:
    """Return the full text of Allen's biography document."""
    with open(DATA_SOURCE, "r", encoding="utf-8") as f:
        return f.read()


llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_MODEL", "gpt-5"),
)
bio_agent = create_agent(
    llm,
    tools=[read_allen_bio],
    system_prompt=(
        "You are an assistant that answers questions about Allen. "
        "Always call the read_allen_bio tool to look up facts before responding, "
        "and say you don't know if the answer isn't in the document."
    ),
)

print("Bio agent created successfully.")

question = "Who is Allen Huan Bui?"
response = bio_agent.invoke({"messages": [{"role": "user", "content": question}]})

answer_content = response["messages"][-1].content
if isinstance(answer_content, list):
    answer_text = "".join(part.get("text", "") for part in answer_content if isinstance(part, dict))
else:
    answer_text = answer_content

print(f"Q: {question}")
print(f"A: {answer_text}")


Bio agent created successfully.
Q: Who is Allen Huan Bui?
A: Allen Huan Bui is a computer science student at the University of Houston (B.S. in Computer Science, expected May 2028; GPA 3.7; Dean’s List Fall 2024 and Spring 2025). He builds AI-assisted software workflows, multi-agent document processing systems, and automation tools, working with Python, LangChain, LangGraph, OpenAI/Azure OpenAI, PyMuPDF, Pydantic, and pandas. His projects include an Agentic Invoice Reconciliation System and an Electricity Agreement Optimizer that leverage multi-agent orchestration and GPT-5. He works part-time at Yomie’s Rice & Yogurt and serves as a Family Leader and Intern with the University of Houston Vietnamese Student Association (Fall 2026–Spring 2027).


## Create Invoice Parser Agent

Create invoice parser agent to extract items from a provided invoice into dataframe

In [3]:
import os
import importlib.util

import fitz
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

load_dotenv()

# invoice-model.py has a hyphen in its filename, so it can't be imported with a normal `import` statement
_spec = importlib.util.spec_from_file_location("invoice_model", "invoice-model - Copy.py")
_invoice_model = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_invoice_model)
Invoice = _invoice_model.Invoice

INVOICE_PDF_PATH = "invoice.pdf"

with fitz.open(INVOICE_PDF_PATH) as doc:
    invoice_text = "\n".join(page.get_text() for page in doc)

llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_MODEL", "gpt-5"),
)

invoice_parser_agent = create_agent(
    llm,
    tools=[],
    name="invoice_parser_agent",
    system_prompt=(
        "You are an invoice parsing assistant. Extract structured invoice data "
        "from the provided document text."
    ),
    response_format=Invoice,
)

result = invoice_parser_agent.invoke({"messages": [{"role": "user", "content": invoice_text}]})
invoice_data: Invoice = result["structured_response"]

invoice_df = pd.DataFrame([item.model_dump() for item in invoice_data.items])
print(invoice_df)

   quantity           description  unit_price    total
0       1.0           Coil Tubing       450.0  45000.0
1       1.0   Full Size Excavator       130.0   1300.0
2       1.0  Large Air Compressor        85.0    850.0


## Create Purchase Order Agent
Create purchase order parser agent to extract items from a provided purchase order document into dataframe


In [4]:
import os
import importlib.util

import fitz
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

load_dotenv()

# purchase-order-model.py has a hyphen in its filename, so it can't be imported with a normal `import` statement
_spec = importlib.util.spec_from_file_location("purchase_order_model", "purchase-order-model - Copy.py")
_purchase_order_model = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_purchase_order_model)
PurchaseOrder = _purchase_order_model.PurchaseOrder

PURCHASE_ORDER_PDF_PATH = "purchase_order.pdf"

with fitz.open(PURCHASE_ORDER_PDF_PATH) as doc:
    purchase_order_text = "\n".join(page.get_text() for page in doc)

llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_MODEL", "gpt-5"),
)

purchase_order_parser_agent = create_agent(
    llm,
    tools=[],
    name="purchase_order_parser_agent",
    system_prompt=(
        "You are a purchase order parsing assistant. Extract structured purchase "
        "order data from the provided document text."
    ),
    response_format=PurchaseOrder,
)

result = purchase_order_parser_agent.invoke({"messages": [{"role": "user", "content": purchase_order_text}]})
purchase_order_data: PurchaseOrder = result["structured_response"]

purchase_order_df = pd.DataFrame([item.model_dump() for item in purchase_order_data.items])
print(purchase_order_df)

   item_no           description  quantity  unit_price  line_total
0        1           Coil Tubing       1.0       450.0     45000.0
1        2   Full Size Excavator       1.0       130.0      1300.0
2        3  Large Air Compressor       1.0        85.0       850.0


## Create Goods Receipt Note Parcer Agent

Create goods receipt note parser agent to extract items from a provided goods receipt note document into dataframe

In [5]:
import os
import importlib.util

import fitz
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

load_dotenv()

# goods-receipt-note-model.py has a hyphen in its filename, so it can't be imported with a normal `import` statement
_spec = importlib.util.spec_from_file_location("goods_receipt_note_model", "goods-receipt-note-model - Copy.py")
_goods_receipt_note_model = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_goods_receipt_note_model)
GoodsReceiptNote = _goods_receipt_note_model.GoodsReceiptNote

GOODS_RECEIPT_NOTE_PDF_PATH = "goods_receipt_note.pdf"

with fitz.open(GOODS_RECEIPT_NOTE_PDF_PATH) as doc:
    goods_receipt_note_text = "\n".join(page.get_text() for page in doc)

llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_MODEL", "gpt-5"),
)

goods_receipt_note_parser_agent = create_agent(
    llm,
    tools=[],
    name="goods_receipt_note_parser_agent",
    system_prompt=(
        "You are a goods receipt note parsing assistant. Extract structured goods "
        "receipt note data from the provided document text."
    ),
    response_format=GoodsReceiptNote,
)

result = goods_receipt_note_parser_agent.invoke({"messages": [{"role": "user", "content": goods_receipt_note_text}]})
goods_receipt_note_data: GoodsReceiptNote = result["structured_response"]

goods_receipt_note_df = pd.DataFrame([item.model_dump() for item in goods_receipt_note_data.items])
print(goods_receipt_note_df)

   item_no           description  quantity_ordered  quantity_received  \
0        1           Coil Tubing               1.0                1.0   
1        2   Full Size Excavator               1.0                1.0   
2        3  Large Air Compressor               1.0                1.0   

   unit_price  line_total        condition_remarks  
0       450.0     45000.0  Received as per invoice  
1       130.0      1300.0  Received as per invoice  
2        85.0       850.0  Received as per invoice  


## Create Contract Parcer Agent

Create contract parser agent to extract items from a provided contract document into dataframe

In [6]:
import os
import importlib.util

import fitz
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

load_dotenv()

# contract-model.py has a hyphen in its filename, so it can't be imported with a normal `import` statement
_spec = importlib.util.spec_from_file_location("contract_model", "contract-model - Copy.py")
_contract_model = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_contract_model)
Contract = _contract_model.Contract

CONTRACT_PDF_PATH = "contract.pdf"

with fitz.open(CONTRACT_PDF_PATH) as doc:
    contract_text = "\n".join(page.get_text() for page in doc)

llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_MODEL", "gpt-5"),
)

contract_parser_agent = create_agent(
    llm,
    tools=[],
    name="contract_parser_agent",
    system_prompt=(
        "You are a contract parsing assistant. Extract structured contract data "
        "from the provided document text."
    ),
    response_format=Contract,
)

result = contract_parser_agent.invoke({"messages": [{"role": "user", "content": contract_text}]})
contract_data: Contract = result["structured_response"]

contract_df = pd.DataFrame([item.model_dump() for item in contract_data.items])
print(contract_df)

   item_no      item_description  quantity  unit_price  extended_amount
0        1           Coil Tubing       1.0       450.0          45000.0
1        2   Full Size Excavator       1.0       130.0           1300.0
2        3  Large Air Compressor       1.0        85.0            850.0


## 6. Create Supervisor Agent Orchestrator/Coordinator

Build a suppervisor agent to orchestrate purchase_order_parcer_agent, invoice_parcer_agent, goods_receipt_note_parcer_agent, and contract_parcer_agent.  This agent should validate or compare the line items from purchase order, invoice, goods receipt note, and contract to ensure all line items are match and align with contract regard to quantity, price.

In [7]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph_supervisor import create_supervisor

load_dotenv()

supervisor_llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_MODEL", "gpt-5"),
)

supervisor_workflow = create_supervisor(
    agents=[
        purchase_order_parser_agent,
        invoice_parser_agent,
        goods_receipt_note_parser_agent,
        contract_parser_agent,
    ],
    model=supervisor_llm,
    prompt=(
        "You are a supervisor coordinating four specialized document-parsing agents: "
        "purchase_order_parser_agent, invoice_parser_agent, goods_receipt_note_parser_agent, "
        "and contract_parser_agent. The full text of each document is included in the "
        "conversation below.\n\n"
        "Delegate to each agent so every document gets parsed into its structured line "
        "items, then compare the resulting line items across all four documents. For each "
        "item, validate that the quantity and unit price match and are consistent with the "
        "contract's price list. Report any discrepancies (missing items, quantity mismatches, "
        "or price mismatches), and confirm which items fully reconcile across all documents."
    ),
)

supervisor_agent = supervisor_workflow.compile()

validation_request = (
    "PURCHASE ORDER DOCUMENT:\n" + purchase_order_text + "\n\n"
    "INVOICE DOCUMENT:\n" + invoice_text + "\n\n"
    "GOODS RECEIPT NOTE DOCUMENT:\n" + goods_receipt_note_text + "\n\n"
    "CONTRACT DOCUMENT:\n" + contract_text + "\n\n"
    "Please validate that the line items (quantity and price) match and align across all four documents."
)

supervisor_result = supervisor_agent.invoke({"messages": [{"role": "user", "content": validation_request}]})

final_content = supervisor_result["messages"][-1].content
if isinstance(final_content, list):
    final_text = "".join(part.get("text", "") for part in final_content if isinstance(part, dict))
else:
    final_text = final_content

print(final_text)

Here’s the cross-document reconciliation of line items after parsing each document.

Parsed items (consistent across all four documents)
- Items present in all: Coil Tubing; Full Size Excavator; Large Air Compressor
- Quantities: 1 for each item in PO, Invoice, GRN (ordered/received), and Contract
- Unit prices: Coil Tubing $450.00; Full Size Excavator $130.00; Large Air Compressor $85.00 across PO, Invoice, GRN, and Contract
- Subtotal/Taxes/Total: Subtotal $47,150.00; Sales Tax $250.00; Shipping $0.00; Total $47,400.00 across all documents

Per-item reconciliation vs contract price list
- Coil Tubing
  - Qty: PO 1 | INV 1 | GRN ordered 1 / received 1 | Contract 1 → MATCH
  - Unit price: PO/INV/GRN/Contract $450.00 → MATCH
  - Line total shown across docs: $45,000.00 (same as Contract extended amount)
  - Calculation check: 1 × $450.00 ≠ $45,000.00 → arithmetic discrepancy
- Full Size Excavator
  - Qty: PO 1 | INV 1 | GRN ordered 1 / received 1 | Contract 1 → MATCH
  - Unit price: PO/